Testing models - ViT5 ft for QA task - https://huggingface.co/PhucDanh/vit5-fine-tuning-for-question-answering (fundamentally generative, but ft'd for qa span head so extractive)

BartPho ft model for QA - https://huggingface.co/PhucDanh/Bartpho-fine-tuning-model-for-question-answering?library=transformers (fundamentally generative, but ft'd for qa span head so extractive)

XLMR-large-viquad - https://huggingface.co/NamSyntax/xlmr-large-viquad (extractive)

phobert-ft-viquad - https://huggingface.co/heidiie/phobert_finetuned_viquad (extractive)

In [ ]:
import json
import time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
)

In [ ]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "NamSyntax/xlmr-large-viquad"
TEST_FILE = "data/input/rq2/context_mapping_phoasr_VIQUAD_val.json" #Swap out the file to run the model on, see readme.txt to choose the corresponding files
OUTPUT_FILE = "data/output/rq2/slide/xlmr_predictions_PhoASR_sliding.json"

In [ ]:
def predict_answer(
    question,
    context,
    debug=False
):
    """

    Uses sliding-window inference so long contexts are
    split into overlapping chunks.

    Returns:
        (best_answer, best_score)
    """

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    max_length = 256
    doc_stride = 128
    max_answer_length = 64
    top_k = 20

    # --------------------------------------------------------
    # Tokenize question separately
    # --------------------------------------------------------

    question_ids = tokenizer(
        question,
        add_special_tokens=False
    )["input_ids"]

    # --------------------------------------------------------
    # Tokenize context separately
    # --------------------------------------------------------

    context_ids = tokenizer(
        context,
        add_special_tokens=False
    )["input_ids"]

    if debug:
        print("Question tokens:", len(question_ids))
        print("Context tokens:", len(context_ids))

    test_pair = tokenizer(
        question,
        "",
        truncation=False,
        add_special_tokens=True
    )["input_ids"]

    question_only = tokenizer(
        question,
        add_special_tokens=False
    )["input_ids"]

    special_overhead = (
        len(test_pair)
        - len(question_only)
    )

    # Maximum number of context tokens per window.
    context_window_size = (
        max_length
        - len(question_ids)
        - special_overhead
    )

    if context_window_size <= 0:
        raise ValueError(
            "Question is too long for max_length="
            f"{max_length}"
        )

    if debug:
        print(
            "Context window size:",
            context_window_size
        )
        print(
            "Doc stride:",
            doc_stride
        )

    # --------------------------------------------------------
    # Create sliding windows over context
    # --------------------------------------------------------

    windows = []

    start = 0

    while start < len(context_ids):

        end = min(
            start + context_window_size,
            len(context_ids)
        )

        window_ids = context_ids[start:end]

        windows.append({
            "ids": window_ids,
            "context_start": start,
            "context_end": end
        })

        # Finished
        if end >= len(context_ids):
            break

        # Move forward while retaining overlap
        next_start = end - doc_stride

        # Safety against infinite loops
        if next_start <= start:
            next_start = start + 1

        start = next_start

    if debug:
        print("Number of windows:", len(windows))

    # --------------------------------------------------------
    # Evaluate every window
    # --------------------------------------------------------

    best_answer = ""
    best_score = float("-inf")
    best_window = None

    for window_index, window in enumerate(windows):

        window_context_ids = window["ids"]

        # ----------------------------------------------------
        # Construct question + context pair
        # ----------------------------------------------------

        window_context = tokenizer.decode(
            window_context_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        inputs = tokenizer(
            question,
            window_context,
            truncation="only_second",
            max_length=max_length,
            return_tensors="pt",
            return_attention_mask=True
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        actual_length = inputs["input_ids"].shape[1]

        if debug:
            print(
                f"Window {window_index + 1}/{len(windows)}:",
                f"context tokens {window['context_start']}"
                f"-{window['context_end']}",
                f"input length={actual_length}"
            )

        # ----------------------------------------------------
        # Model inference
        # ----------------------------------------------------

        with torch.no_grad():
            outputs = model(**inputs)

        start_logits = outputs.start_logits[0]
        end_logits = outputs.end_logits[0]

        # ----------------------------------------------------
        # Locate the context in the pair
        #
        # Since we are using a slow tokenizer, don't use
        # sequence_ids().
        #
        # Find the window token sequence inside input_ids.
        # ----------------------------------------------------

        input_ids = inputs["input_ids"][0].tolist()

        window_ids = tokenizer(
            window_context,
            add_special_tokens=False
        )["input_ids"]

        context_start = None

        for i in range(
            len(input_ids) - len(window_ids) + 1
        ):

            if input_ids[
                i:i + len(window_ids)
            ] == window_ids:

                context_start = i
                break

        if context_start is None:

            if debug:
                print(
                    "Could not locate context "
                    f"for window {window_index}"
                )

            continue

        context_end = (
            context_start
            + len(window_ids)
            - 1
        )

        # ----------------------------------------------------
        # Restrict logits to context
        # ----------------------------------------------------

        context_start_logits = start_logits[
            context_start:
            context_end + 1
        ]

        context_end_logits = end_logits[
            context_start:
            context_end + 1
        ]

        # ----------------------------------------------------
        # Get top candidate positions
        # ----------------------------------------------------

        current_top_k = min(
            top_k,
            len(context_start_logits)
        )

        start_values, start_indices = torch.topk(
            context_start_logits,
            current_top_k
        )

        end_values, end_indices = torch.topk(
            context_end_logits,
            current_top_k
        )

        # ----------------------------------------------------
        # Search candidate spans
        # ----------------------------------------------------

        for start_value, start_idx in zip(
            start_values,
            start_indices
        ):

            for end_value, end_idx in zip(
                end_values,
                end_indices
            ):

                start_idx = int(start_idx)
                end_idx = int(end_idx)

                # Invalid span
                if end_idx < start_idx:
                    continue

                # Answer too long
                if (
                    end_idx
                    - start_idx
                    + 1
                    > max_answer_length
                ):
                    continue

                # Convert local positions to positions
                # in the complete model input.
                start_position = (
                    context_start
                    + start_idx
                )

                end_position = (
                    context_start
                    + end_idx
                )

                answer_ids = inputs["input_ids"][
                    0,
                    start_position:
                    end_position + 1
                ]

                answer = tokenizer.decode(
                    answer_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                ).strip()

                if not answer:
                    continue

                score = (
                    float(start_value)
                    + float(end_value)
                )

                # ------------------------------------------------
                # Keep globally best answer
                # ------------------------------------------------

                if score > best_score:

                    best_score = score
                    best_answer = answer
                    best_window = window_index

    # --------------------------------------------------------
    # No answer found
    # --------------------------------------------------------

    if best_score == float("-inf"):

        if debug:
            print("No valid answer found.")

        return "", 0.0

    if debug:
        print(
            "Best window:",
            best_window
        )
        print(
            "Best answer:",
            best_answer
        )
        print(
            "Best score:",
            best_score
        )

    return best_answer, best_score

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print(f"{MODEL_NAME} ViQuAD Inference")
print("=" * 70)

print(f"Model : {MODEL_NAME}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

print("=" * 70)


# ============================================================
# Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

print("[OK] Tokenizer loaded.")


# ============================================================
# Load model
# ============================================================

print("\n[2/4] Loading model...")

model = AutoModelForQuestionAnswering.from_pretrained(
    MODEL_NAME
)

model.to(device)
model.eval()

print("[OK] Model loaded.")


# ============================================================
# Load test data
# ============================================================

print("\n[3/4] Loading test data...")

with open(TEST_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

results = data.get("results", [])

print(f"[OK] Loaded {len(results):,} contexts.")


# ============================================================
# Count valid questions
# ============================================================

total_questions = 0
skipped_contexts = 0
skipped_questions = 0

for result in results:

    context_id = result.get(
        "context_id",
        ""
    )

    transcription = result.get(
        "transcription",
        ""
    )

    if not transcription:
        continue

    for qa in result.get("qas", []):

        question = qa.get(
            "question",
            ""
        )

        answers = qa.get(
            "answers",
            []
        )

        if not question or not answers:
            continue

        reference_answer = answers[0]

        answer_text = reference_answer.get(
            "text",
            ""
        )

        original_answer_start = (
            reference_answer.get(
                "answer_start",
                -1
            )
        )

        if not answer_text:
            continue


    context = result.get("transcription", "")

    if not context.strip():
        skipped_contexts += 1
        continue

    for qa in result.get("qas", []):

        question = qa.get("question", "")

        if question.strip():
            total_questions += 1
        else:
            skipped_questions += 1


print(f"[INFO] Valid questions : {total_questions:,}")
print(f"[INFO] Empty contexts  : {skipped_contexts:,}")
print(f"[INFO] Empty questions : {skipped_questions:,}")

NamSyntax/xlmr-large-viquad ViQuAD Inference
Model : NamSyntax/xlmr-large-viquad
Device: cpu
[OK] Tokenizer loaded.

[2/4] Loading model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[OK] Model loaded.

[3/4] Loading test data...
[OK] Loaded 557 contexts.
[INFO] Valid questions : 3,814
[INFO] Empty contexts  : 0
[INFO] Empty questions : 0


In [ ]:
# ============================================================
# Run inference
# ============================================================

print("\n[4/4] Starting inference...")
print("=" * 70)

predictions = []

processed_questions = 0
successful_questions = 0
failed_questions = 0

start_time = time.time()


for context_idx, result in enumerate(results):

    context_id = result.get(
        "context_id",
        f"context_{context_idx}"
    )

    context = result.get(
        "transcription",
        ""
    )

    qas = result.get(
        "qas",
        []
    )

    print(
        f"\n[CONTEXT {context_idx + 1}/{len(results)}]"
        f" ID={context_id}"
        f" | questions={len(qas)}"
        f" | chars={len(context):,}"
    )

    # --------------------------------------------------------
    # Empty context
    # --------------------------------------------------------

    if not context.strip():

        print(
            "  [WARNING] Empty transcription - skipping."
        )

        continue

    # --------------------------------------------------------
    # Questions
    # --------------------------------------------------------

    for qa_idx, qa_item in enumerate(qas):

        question_id = qa_item.get(
            "id",
            f"{context_id}-{qa_idx}"
        )

        question = qa_item.get(
            "question",
            ""
        )

        # Empty question
        if not question.strip():

            print(
                f"  [{question_id}] "
                f"[WARNING] Empty question - skipping."
            )

            continue

        processed_questions += 1

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        elapsed = time.time() - start_time

        rate = (
            processed_questions / elapsed
            if elapsed > 0
            else 0
        )

        remaining = (
            total_questions - processed_questions
        )

        eta = (
            remaining / rate
            if rate > 0
            else 0
        )

        print(
            f"  [{processed_questions:,}/{total_questions:,}] "
            f"{processed_questions / total_questions * 100:6.2f}% "
            f"| ID={question_id} "
            f"| {rate:.2f} q/s "
            f"| ETA={eta / 60:.1f} min"
        )

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        try:
            answer, score = predict_answer(
                question,
                context
            )

            successful_questions += 1

            predictions.append({
                "context_id": context_id,
                "question_id": question_id,
                "question": question,
                "prediction": answer,
                "score": score,
            })

            print(
                f"      Answer: {answer!r}"
                f" | score={score:.4f}"
            )

        except Exception as e:

            failed_questions += 1

            print(
                f"      [ERROR] "
                f"{type(e).__name__}: {e}"
            )

            predictions.append({
                "context_id": context_id,
                "question_id": question_id,
                "question": question,
                "prediction": "",
                "score": 0.0,
                "error": str(e),
            })


# ============================================================
# Save predictions
# ============================================================

print("\n" + "=" * 70)
print("Saving predictions...")

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# Final statistics
# ============================================================

total_time = time.time() - start_time

print("=" * 70)
print("DONE")
print("=" * 70)

print(
    f"Questions processed : {processed_questions:,}"
)

print(
    f"Successful           : {successful_questions:,}"
)

print(
    f"Failed               : {failed_questions:,}"
)

print(
    f"Total time           : "
    f"{total_time / 60:.2f} minutes"
)

if processed_questions > 0:

    print(
        f"Average time/question: "
        f"{total_time / processed_questions:.3f} sec"
    )

    print(
        f"Throughput           : "
        f"{processed_questions / total_time:.2f} questions/sec"
    )

print(
    f"Predictions saved to : {OUTPUT_FILE}"
)

print("=" * 70)


[4/4] Starting inference...

[CONTEXT 1/557] ID=0006-0017 | questions=6 | chars=776
  [1/3,814]   0.03% | ID=0006-0017-0001 | 1618.80 q/s | ETA=0.0 min
      Answer: 'Nhiều khách sạn các khu nghỉ dưỡng, trung tâm hội thảo lớn như trung tâm hội nghị Pua El Torico' | score=21.1629
  [2/3,814]   0.05% | ID=0006-0017-0002 | 0.66 q/s | ETA=96.4 min
      Answer: 'ngành hóa dầu, dược phẩm và khoa học công nghệ du lịch' | score=18.7217
  [3/3,814]   0.08% | ID=0006-0017-0003 | 0.53 q/s | ETA=118.9 min
      Answer: 'năm hai nghìn không trăm linh bảy' | score=10.3507
  [4/3,814]   0.10% | ID=0006-0017-0004 | 0.48 q/s | ETA=131.1 min
      Answer: 'nông nghiệp' | score=16.1299
  [5/3,814]   0.13% | ID=0006-0017-0005 | 0.43 q/s | ETA=146.9 min
      Answer: 'trồng mía đường' | score=-11.9357
  [6/3,814]   0.16% | ID=0006-0017-0006 | 0.39 q/s | ETA=164.3 min
      Answer: 'ngành hóa dầu, dược phẩm và khoa học công nghệ du lịch' | score=13.6536

[CONTEXT 2/557] ID=0004-0030 | questions=7 | chars=

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (688 > 512). Running this sequence through the model will result in indexing errors


      Answer: 'Hai ông bà sau có 4 con trai Harlan Gandhi, sinh năm 1888, Maniland Gandhi, sinh năm 1892, Ramda Gandhi, sinh năm 1897 và Diodat Gandhi, sinh năm 1900' | score=-13.3544

[CONTEXT 270/557] ID=0017-0041 | questions=8 | chars=2,887
  [1,841/3,814]  48.27% | ID=0017-0041-0001 | 0.23 q/s | ETA=141.3 min
      Answer: 'người Pháp đã đề nghị nhượng lại tất cả các thuộc địa của họ tại vùng biển Caribe và Thái Bình Dương với giá mười tỷ franc đánh đổi bằng quyền mua theo tín dụng không giới hạn các phi cơ Mỹ' | score=-7.1698
  [1,842/3,814]  48.30% | ID=0017-0041-0002 | 0.23 q/s | ETA=141.5 min
      Answer: 'rằng ông chỉ đang theo đuổi chính sách ngoại giao biệt lập mà sẽ không đẩy Hoa Kỳ vào bất cứ một cuộc chiến nào' | score=-16.5381
  [1,843/3,814]  48.32% | ID=0017-0041-0003 | 0.23 q/s | ETA=141.7 min
      Answer: 'rằng ông chỉ đang theo đuổi chính sách ngoại giao biệt lập mà sẽ không đẩy Hoa Kỳ vào bất cứ một cuộc chiến nào' | score=-14.2985
  [1,844/3,814]  48.35% | ID=00

EM & F1 EVAL

In [ ]:
import json
import re
import string
from collections import Counter


# ============================================================
# Configuration
# ============================================================

PREDICTIONS_FILE = "data/output/rq2/slide/xlmr_predictions_PhoASR_sliding.json"
GROUND_TRUTH_FILE = "data/input/rq2/context_mapping_phoasr_VIQUAD_val.json"

EVALUATION_FILE = "data/output/rq2/slide/xlmr_evaluation_PhoASR_sliding.json"


# ============================================================
# SQuAD-style normalization
# ============================================================

def normalize_answer(text):
    """
    SQuAD-style normalization:
    - lowercase
    - remove punctuation
    - remove English articles
    - normalize whitespace
    """

    def remove_articles(text):
        return re.sub(
            r"\b(a|an|the)\b",
            " ",
            text
        )

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(
            ch for ch in text
            if ch not in exclude
        )

    def lower(text):
        return text.lower()

    return white_space_fix(
        remove_articles(
            remove_punc(
                lower(text)
            )
        )
    )


# ============================================================
# Exact Match
# ============================================================

def exact_match_score(
    prediction,
    ground_truth
):
    return int(
        normalize_answer(prediction)
        == normalize_answer(ground_truth)
    )


# ============================================================
# F1
# ============================================================

def f1_score(
    prediction,
    ground_truth
):

    prediction_tokens = normalize_answer(
        prediction
    ).split()

    ground_truth_tokens = normalize_answer(
        ground_truth
    ).split()

    # Both are empty
    if (
        len(prediction_tokens) == 0
        and len(ground_truth_tokens) == 0
    ):
        return 1.0

    # One is empty
    if (
        len(prediction_tokens) == 0
        or len(ground_truth_tokens) == 0
    ):
        return 0.0

    common = (
        Counter(prediction_tokens)
        & Counter(ground_truth_tokens)
    )

    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = (
        num_same
        / len(prediction_tokens)
    )

    recall = (
        num_same
        / len(ground_truth_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ============================================================
# Load predictions
# ============================================================

print("=" * 70)
print("ViQuAD EM / F1 Evaluation")
print("=" * 70)

print("\nLoading predictions...")

with open(
    PREDICTIONS_FILE,
    "r",
    encoding="utf-8"
) as f:

    predictions = json.load(f)

print(
    f"Loaded {len(predictions):,} predictions."
)


# ============================================================
# Load ground truth
# ============================================================

print("\nLoading ground truth...")

with open(
    GROUND_TRUTH_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

results = data["results"]

print(
    f"Loaded {len(results):,} contexts."
)


# ============================================================
# Build ground-truth dictionary
# ============================================================

ground_truth = {}

empty_answer_questions = []
impossible_questions = []

for result in results:

    for qa in result.get("qas", []):

        question_id = qa.get("id")

        answers = qa.get(
            "answers",
            []
        )

        is_impossible = qa.get(
            "is_impossible",
            False
        )

        # ----------------------------------------------------
        # Extract answer texts
        # ----------------------------------------------------

        answer_texts = [
            answer.get("text", "")
            for answer in answers
            if answer.get("text", "") is not None
        ]

        # Remove completely empty answers
        answer_texts = [
            answer
            for answer in answer_texts
            if answer.strip() != ""
        ]

        ground_truth[question_id] = answer_texts

        # ----------------------------------------------------
        # Keep track of problematic examples
        # ----------------------------------------------------

        if is_impossible:
            impossible_questions.append(
                question_id
            )

        elif len(answer_texts) == 0:
            empty_answer_questions.append(
                question_id
            )


# ============================================================
# Dataset statistics
# ============================================================

print("\nGround-truth statistics:")
print(
    f"  Questions with answers : "
    f"{len(ground_truth) - len(empty_answer_questions):,}"
)

print(
    f"  Empty answer lists     : "
    f"{len(empty_answer_questions):,}"
)

print(
    f"  Impossible questions   : "
    f"{len(impossible_questions):,}"
)


# ============================================================
# Calculate EM and F1
# ============================================================

total = 0
em_total = 0.0
f1_total = 0.0

missing_ground_truth = 0
skipped_empty_ground_truth = 0

per_question_results = []


for prediction_item in predictions:

    question_id = prediction_item.get(
        "question_id"
    )

    prediction = prediction_item.get(
        "prediction",
        ""
    )

    # --------------------------------------------------------
    # Find ground truth
    # --------------------------------------------------------

    if question_id not in ground_truth:

        missing_ground_truth += 1

        print(
            f"[WARNING] No ground truth for "
            f"{question_id}"
        )

        continue

    references = ground_truth[
        question_id
    ]

    # --------------------------------------------------------
    # Handle empty ground truth
    # --------------------------------------------------------

    if len(references) == 0:

        skipped_empty_ground_truth += 1

        continue

    # --------------------------------------------------------
    # Calculate against every valid reference
    # and take the best score
    # --------------------------------------------------------

    em = max(
        exact_match_score(
            prediction,
            reference
        )
        for reference in references
    )

    f1 = max(
        f1_score(
            prediction,
            reference
        )
        for reference in references
    )

    em_total += em
    f1_total += f1
    total += 1

    per_question_results.append({
        "question_id": question_id,
        "prediction": prediction,
        "ground_truth": references,
        "em": em,
        "f1": f1
    })


# ============================================================
# Final metrics
# ============================================================

if total > 0:

    em = (
        em_total / total
    ) * 100

    f1 = (
        f1_total / total
    ) * 100

else:

    em = 0.0
    f1 = 0.0


# ============================================================
# Print results
# ============================================================

print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)

print(
    f"Prediction records       : "
    f"{len(predictions):,}"
)

print(
    f"Evaluated questions      : "
    f"{total:,}"
)

print(
    f"Skipped empty ground truth: "
    f"{skipped_empty_ground_truth:,}"
)

print(
    f"Missing ground truth     : "
    f"{missing_ground_truth:,}"
)

print(
    f"Exact Match (EM)         : "
    f"{em:.2f}"
)

print(
    f"F1                       : "
    f"{f1:.2f}"
)

print("=" * 70)


# ============================================================
# Save detailed evaluation
# ============================================================

evaluation = {
    "num_predictions": len(predictions),
    "num_evaluated": total,
    "num_skipped_empty_ground_truth":
        skipped_empty_ground_truth,
    "num_missing_ground_truth":
        missing_ground_truth,
    "exact_match": em,
    "f1": f1,
    "questions": per_question_results
}

with open(
    EVALUATION_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"\nDetailed results saved to: "
    f"{EVALUATION_FILE}"
)



ViQuAD EM / F1 Evaluation

Loading predictions...
Loaded 3,814 predictions.

Loading ground truth...
Loaded 557 contexts.

Ground-truth statistics:
  Questions with answers : 3,771
  Empty answer lists     : 0
  Impossible questions   : 1,161

RESULTS
Prediction records       : 3,814
Evaluated questions      : 2,653
Skipped empty ground truth: 1,161
Missing ground truth     : 0
Exact Match (EM)         : 24.20
F1                       : 52.86

Detailed results saved to: /content/drive/MyDrive/data/xlmr_evaluation_PhoASR_sliding.json
